[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# The Index &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

In [1]:
import math
from datetime import datetime, timedelta

import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


**1.** Make a table of the stations indexed by name, and print Kirkenes's latitude with `loc`.

In [2]:
by_name = stations.set_index("station")

print(by_name.loc["Kirkenes", "latitude"])

69.73


Kirkenes has a row in the stations table although it has no readings, so the lookup finds it.

**2.** Make Tromso's readings a Series indexed by time, and show its readings on 9 July from 13:00
to 17:00.

In [3]:
tromso = readings[readings["station"] == "Tromso"].set_index("time")["celsius"]

tromso.loc["2025-07-09 13:00":"2025-07-09 17:00"]

time
2025-07-09 13:00:00    15.2
2025-07-09 14:00:00    15.8
2025-07-09 15:00:00    16.2
2025-07-09 16:00:00    14.7
2025-07-09 17:00:00    14.7
Name: celsius, dtype: float64

A slice by label includes both ends, so 13:00 and 17:00 are both in the result, five hours in all.

**3.** From the same Series, print how many readings 9 July has and their mean, rounded to two
places.

In [4]:
tromso = readings[readings["station"] == "Tromso"].set_index("time")["celsius"]
july_9 = tromso.loc["2025-07-09"]

print(len(july_9), round(july_9.mean(), 2))

24 12.35


The date on its own selects every hour of that day. The first line rebuilds the Series, so that this
answer runs without the one before it.

**4.** Turn Tromso's Series back into a table with `reset_index`, and show its first three rows.

In [5]:
tromso = readings[readings["station"] == "Tromso"].set_index("time")["celsius"]

tromso.reset_index().head(3)

,time,celsius
0,2025-01-01 00:00:00,-6.7
1,2025-01-01 01:00:00,-8.5
2,2025-01-01 02:00:00,-8.5


The times come back as a column named `time`, and the rows are numbered from 0 again.

**5.** With a sorted station and time MultiIndex, show every station's reading at 03:00 on 8
January, using `xs`.

In [6]:
in_order = readings.set_index(["station", "time"]).sort_index()

in_order.xs("2025-01-08 03:00", level="time")

,celsius
station,
Bergen,-4.4
Oslo,-6.0
Svalbard,-17.1
Tromso,-9.2


`xs` keeps every station and the one time.

---

&#8592; **Back to:** [The Index](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/09-the-index.ipynb)  &nbsp;&middot;&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)
